## Animated Exoplanet Discovery Plot

This notebook generates an animated **mass vs orbital period** diagram of confirmed exoplanets using the NASA Exoplanet Archive `PSCompPars` table.

The animation progresses through **discovery year**, showing how the population of known exoplanets has grown over time. Each frame adds newly discovered planets cumulatively.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.lines import Line2D


## User Options

Several parameters can be adjusted to customise the animation.


### Animation Settings

| Parameter | Description |
|------|------|
| `FPS` | Frames per second in the GIF |
| `POINT_SIZE` | Size of the planet markers |
| `ALPHA` | Transparency of the markers |

### Visual Style

| Option | Description |
|------|------|
| `THEME="light" / "dark"` | White/black text and white/black plot background |
| `TRANSPARENT_BG=True/False` | Entire plot background is transparent or not |
| `GRIDLINES=True/False` | Enables/Disables background grid |
| `LOG_SCALE=True/False` | Logarithmic axes (recommended) or Linear axes |

### Axis Units

These control the physical units used in the plot.

| Option | Description |
|------|------|
| `X_SCALE="days"/"years"` | Orbital period shown in days/years |
| `Y_SCALE="Mearth"/"Mjup"` | Planet mass in Earth/Jupiter masses |

### Output
The animation will be saved as a GIF suitable for slides, websites, or outreach presentations.

In [28]:
# USER OPTIONS

CSV_PATH = "PSCompPars_2026.03.03_07.46.08.csv"
OUT_GIF = "mass_vs_period_by_year.gif"

FPS = 1
POINT_SIZE = 9
ALPHA = 0.85
FIGSIZE = (11.8, 6.6)    # has to be wider to make room for legend on the side
DPI = 500

THEME = "dark"           # "light" or "dark"
GRIDLINES = False
TRANSPARENT_BG = False
LOG_SCALE = True

# Axis unit choices
X_SCALE = "years"         # "days" or "years"
Y_SCALE = "Mjup"       # "Mearth" or "Mjup"

# Method choides + colours
METHOD_COLOR = {
    "Direct imaging": "#E4768F",
    "Microlensing":   "#E6AD7F",
    "Transit":        "#9BE8C9",
    "Radial velocity":"#8EA3E8",
}

METHOD_ORDER = ["Transit", "Radial velocity", "Microlensing", "Direct imaging"]

# Map raw strings in the CSV to four detection methods
# (Add any variations you see in file pls.)
METHOD_SYNONYMS = {
    "Direct imaging": {"Imaging", "Direct Imaging", "Direct imaging"},
    "Microlensing":   {"Microlensing"},
    "Transit":        {"Transit", "Transits"},
    "Radial velocity":{"Radial Velocity", "Radial velocity", "RV", "Radial Velocity (RV)"},
}


In [29]:
# RGBA -> GIF palette frame with transparency preserved
def rgba_to_gif_frame(rgba: Image.Image, transparent_index: int = 255) -> Image.Image:
    """
    Convert an RGBA Pillow Image to a GIF-friendly 'P' image, preserving transparency.
    We:
      1) quantize to <=255 colors in RGB (reserve 1 palette slot for transparency),
      2) force all fully transparent pixels to a chosen palette index,
      3) set that index as transparent in .info['transparency'].
    """
    if rgba.mode != "RGBA":
        rgba = rgba.convert("RGBA")

    alpha = rgba.getchannel("A")
    transparent_mask = np.array(alpha) == 0

    # Quantise RGB
    rgb = rgba.convert("RGB")
    p = rgb.quantize(colors=255, method=Image.Quantize.FASTOCTREE)  # 255 colors (0..254)
    p = p.convert("P")  

    palette = p.getpalette()
    if palette is None:
        palette = [0] * (256 * 3)
    if len(palette) < 256 * 3:
        palette = palette + [0] * (256 * 3 - len(palette))

    palette[transparent_index * 3 + 0] = 0
    palette[transparent_index * 3 + 1] = 255
    palette[transparent_index * 3 + 2] = 0
    p.putpalette(palette)

    arr = np.array(p, dtype=np.uint8)
    arr[transparent_mask] = transparent_index
    p = Image.fromarray(arr, mode="P")
    p.putpalette(palette)

    p.info["transparency"] = transparent_index
    return p



# Theme styling (text/spines only; bg is transparent)
THEME = THEME.lower().strip()
if THEME not in {"light", "dark"}:
    raise ValueError("THEME must be 'light' or 'dark'")

if THEME == "light":
    textcolor = "black"
    spinecolor = "black"
    grid_alpha = 0.25
else:
    textcolor = "#D0D0D0"
    spinecolor = "#B0B0B0"
    grid_alpha = 0.22



# Read data (skip '# COLUMN ...' comment lines)
df = pd.read_csv(CSV_PATH, comment="#", low_memory=False)

need = ["pl_name", "discoverymethod", "disc_year", "pl_orbper", "pl_bmasse", "pl_bmassj"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

d = df[need].copy()
d["disc_year"] = pd.to_numeric(d["disc_year"], errors="coerce")
d["pl_orbper"] = pd.to_numeric(d["pl_orbper"], errors="coerce")
d["pl_bmasse"] = pd.to_numeric(d["pl_bmasse"], errors="coerce")
d["pl_bmassj"] = pd.to_numeric(d["pl_bmassj"], errors="coerce")

d = d.dropna(subset=["disc_year", "pl_orbper", "discoverymethod"])
# Choose mass column depending on unit
if Y_SCALE == "Mearth":
    mass_col = "pl_bmasse"
elif Y_SCALE == "Mjup":
    mass_col = "pl_bmassj"
else:
    raise ValueError("Y_SCALE must be 'Mearth' or 'Mjup'")

d = d.dropna(subset=[mass_col])
d = d[(d["pl_orbper"] > 0) & (d[mass_col] > 0)]
d["disc_year"] = d["disc_year"].astype(int)

d = d.sort_values("disc_year").drop_duplicates(subset=["pl_name"], keep="first")



# Filter to 4 methods
raw_to_canon = {}
for canon, raws in METHOD_SYNONYMS.items():
    for r in raws:
        raw_to_canon[r] = canon

d["method_group"] = d["discoverymethod"].map(raw_to_canon)
d = d.dropna(subset=["method_group"]).copy()

if len(d) == 0:
    raise ValueError(
        "No planets left after filtering to the 4 selected detection methods.\n"
        "Check df['discoverymethod'].unique() and update METHOD_SYNONYMS."
    )

groups = [m for m in METHOD_ORDER if m in set(d["method_group"].unique())]



# Fix axis limits across all frames
# X-axis scaling
if X_SCALE == "days":
    x = d["pl_orbper"].to_numpy()
    x_label = "Orbital period (days)"
elif X_SCALE == "years":
    x = d["pl_orbper"].to_numpy() / 365.25
    x_label = "Orbital period (years)"
else:
    raise ValueError("X_SCALE must be 'days' or 'years'")

# Y-axis scaling
y = d[mass_col].to_numpy()

if Y_SCALE == "Mearth":
    y_label = "Planet mass (Earth masses)"
elif Y_SCALE == "Mjup":
    y_label = "Planet mass (Jupiter masses)"

x_lo, x_hi = np.quantile(x, [0.001, 0.999])
y_lo, y_hi = np.quantile(y, [0.001, 0.999])

x_lo, x_hi = x_lo / 1.5, x_hi * 1.5
y_lo, y_hi = y_lo / 1.5, y_hi * 1.5

years = np.arange(d["disc_year"].min(), d["disc_year"].max() + 1)
d_sorted = d.sort_values("disc_year")



# Build figure
fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)

# Always keep the OUTSIDE (figure background) transparent
fig.patch.set_alpha(0.0)
fig.set_facecolor((0, 0, 0, 0))

if TRANSPARENT_BG:
    # Plot area also transparent
    ax.patch.set_alpha(0.0)
    ax.set_facecolor((0, 0, 0, 0))
else:
    # Plot area opaque: black in dark mode, white in light mode
    ax.patch.set_alpha(1.0)
    ax.set_facecolor("black" if THEME == "dark" else "white")
fig.subplots_adjust(right=0.78)

scatters = {}
for g in groups:
    scatters[g] = ax.scatter(
        [], [],
        s=POINT_SIZE,
        alpha=ALPHA,
        color=METHOD_COLOR[g],
        edgecolors="none"
    )

if LOG_SCALE:
    ax.set_xscale("log")
    ax.set_yscale("log")
ax.set_xlim(x_lo, x_hi)
ax.set_ylim(y_lo, y_hi)

ax.set_title("Confirmed exoplanets: Mass vs Period (cumulative by discovery year)", color=textcolor)
ax.set_xlabel(x_label, color=textcolor)
ax.set_ylabel(y_label, color=textcolor)

ax.tick_params(colors=textcolor, which="both")
for spine in ax.spines.values():
    spine.set_color(spinecolor)

if GRIDLINES:
    ax.grid(True, which="both", alpha=grid_alpha)
else:
    ax.grid(False)

year_text = ax.text(
    0.98, 0.02, "",
    transform=ax.transAxes,
    ha="right", va="bottom",
    fontsize=16, fontweight="bold",
    color=textcolor
)

handles = [
    Line2D([0], [0], marker='o', linestyle="None",
           markerfacecolor=METHOD_COLOR[g], markeredgecolor="none",
           markersize=9, label=g)
    for g in groups
]
leg = ax.legend(
    handles=handles,
    title="Detection method",
    loc="center left",
    bbox_to_anchor=(1.01, 0.5),
    frameon=False
)
plt.setp(leg.get_title(), color=textcolor)
for t in leg.get_texts():
    t.set_color(textcolor)



# Update + render frames
def update(frame_year: int):
    subset = d_sorted[d_sorted["disc_year"] <= frame_year]
    for g in groups:
        sg = subset[subset["method_group"] == g]
        offsets = np.column_stack([
            sg["pl_orbper"].to_numpy() / 365.25 if X_SCALE == "years" else sg["pl_orbper"].to_numpy(),
            sg[mass_col].to_numpy()
        ])
        scatters[g].set_offsets(offsets)
    year_text.set_text(str(frame_year))

frames = []
duration_ms = int(1000 / FPS)

for yr in years:
    update(int(yr))
    fig.canvas.draw()

    w, h = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8).reshape((h, w, 4))
    rgba = Image.fromarray(buf, mode="RGBA")

    frames.append(rgba_to_gif_frame(rgba, transparent_index=255))

plt.close(fig)

print(f"Frames generated: {len(frames)} | Years: {years.min()}–{years.max()}")

if len(frames) < 2:
    raise RuntimeError(
        "Only one frame generated after filtering. That will look static.\n"
        "This means only one discovery year remains after filtering."
    )



# Save animated GIF with transparency
frames[0].save(
    OUT_GIF,
    save_all=True,
    append_images=frames[1:],
    duration=duration_ms,
    loop=0,
    optimize=False,
    transparency=255, 
    disposal=2
)

print(f"Saved transparent animated GIF: {OUT_GIF}")

Frames generated: 32 | Years: 1995–2026
Saved transparent animated GIF: mass_vs_period_by_year.gif
